<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/GQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import torch
import torch.nn as nn


class Llama3GQA(nn.Module):
    def __init__(self):
        super().__init__()


        self.d_model = 4096
        self.num_q_heads = 32
        self.num_kv_heads = 8
        self.head_dim = self.d_model // self.num_q_heads   # 4096 // 32 = 128
        self.group_size = self.num_q_heads // self.num_kv_heads  # 32 // 8 = 4


        self.Wq = nn.Linear(self.d_model, self.num_q_heads * self.head_dim, bias=False)
        self.Wk = nn.Linear(self.d_model, self.num_kv_heads * self.head_dim, bias=False)
        self.Wv = nn.Linear(self.d_model, self.num_kv_heads * self.head_dim, bias=False)
        self.Wo = nn.Linear(self.num_q_heads * self.head_dim, self.d_model, bias=False)

    def forward(self, x, mask=None):
        """
        x: [B, T, 4096]
        """
        B, T, _ = x.shape


        q = self.Wq(x)
        k = self.Wk(x)
        v = self.Wv(x)   # [B, T, 8*128]  = [B, T, 1024]


        q = q.view(B, T, self.num_q_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)


        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)

        # k: [B, 32, T, 128]
        # v: [B, 32, T, 128]


        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)


        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attn = torch.softmax(scores, dim=-1)

        out = torch.matmul(attn, v)

        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        out = self.Wo(out)


        return out

In [2]:
import torch
torch.manual_seed(42)

model = Llama3GQA()
dummy_input = torch.randn(1, 10, model.d_model)  # Batch size 1, sequence length 10, d_model 4096
output = model(dummy_input)
print("Output shape:", output.shape)

Output shape: torch.Size([1, 10, 4096])
